In [1]:
!pip install datasets transformers scikit-learn

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DefaultDataCollator
import torch
import numpy as np
from sklearn.metrics import f1_score

In [3]:
dataset = load_dataset("go_emotions")

num_labels = 28

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [4]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

dataset = dataset.map(tokenize, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [5]:
def to_multi_hot(example):
    multi_hot = np.zeros(num_labels, dtype=np.float32)
    multi_hot[example["labels"]] = 1.0
    example["labels"] = multi_hot
    return example

dataset = dataset.map(to_multi_hot)

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [6]:
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [7]:
train_dataset = dataset["train"].select(range(5000))
val_dataset = dataset["validation"].select(range(500))

print(len(train_dataset), len(val_dataset))

5000 500


In [8]:
print(dataset["train"][0]["labels"])
print(type(dataset["train"][0]["labels"]))

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1])
<class 'torch.Tensor'>


In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
label_names = dataset["train"].features["labels"].feature.names

for i, name in enumerate(label_names):
    print(i, name)

0 admiration
1 amusement
2 anger
3 annoyance
4 approval
5 caring
6 confusion
7 curiosity
8 desire
9 disappointment
10 disapproval
11 disgust
12 embarrassment
13 excitement
14 fear
15 gratitude
16 grief
17 joy
18 love
19 nervousness
20 optimism
21 pride
22 realization
23 relief
24 remorse
25 sadness
26 surprise
27 neutral


In [11]:
tail_labels = [16, 21, 23, 19, 12, 24, 14, 8]

In [12]:
def tail_recall(labels, preds, tail_labels):
    recalls = []
    for i in tail_labels:
        true = labels[:, i]
        pred = preds[:, i]

        tp = ((true == 1) & (pred == 1)).sum()
        fn = ((true == 1) & (pred == 0)).sum()

        if tp + fn > 0:
            recalls.append(tp / (tp + fn))

    return np.mean(recalls) if recalls else 0


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    tail = tail_recall(labels, preds, tail_labels)

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "tail_recall": tail,
    }

In [13]:
data_collator = DefaultDataCollator(return_tensors="pt")

class MultiLabelCollator:
    def __call__(self, features):
        batch = data_collator(features)
        batch["labels"] = batch["labels"].float()
        return batch

collator = MultiLabelCollator()

In [14]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_steps=50,
    report_to="none",
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=collator,
)

In [16]:
trainer.train()

Step,Training Loss
50,0.372235
100,0.169465
150,0.156164
200,0.151040
250,0.151606
300,0.144572
350,0.143289
400,0.134715
450,0.125797
500,0.120047


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=939, training_loss=0.13806644136024612, metrics={'train_runtime': 178.615, 'train_samples_per_second': 83.98, 'train_steps_per_second': 5.257, 'total_flos': 496983075840000.0, 'train_loss': 0.13806644136024612, 'epoch': 3.0})

In [17]:
trainer.evaluate()

{'eval_loss': 0.1087929755449295,
 'eval_micro_f1': 0.4014510278113664,
 'eval_macro_f1': 0.1311509900895599,
 'eval_tail_recall': 0.0,
 'eval_runtime': 1.8589,
 'eval_samples_per_second': 268.978,
 'eval_steps_per_second': 17.215,
 'epoch': 3.0}

In [18]:
pred_output = trainer.predict(val_dataset)

logits = pred_output.predictions
labels = pred_output.label_ids

probs = 1 / (1 + np.exp(-logits))

print(probs[:5])
print("max prob:", probs.max())
print("mean prob:", probs.mean())

[[0.03557869 0.02509172 0.02675388 0.02721472 0.06824304 0.02835806
  0.20415841 0.46063048 0.02201623 0.02793692 0.04020547 0.02066313
  0.01296328 0.03921309 0.01749409 0.0233234  0.0191868  0.02275692
  0.02415852 0.01658568 0.04989529 0.0061833  0.04200246 0.0163744
  0.01975982 0.03413587 0.0937056  0.1612869 ]
 [0.01332541 0.00691502 0.00983415 0.02805346 0.07062496 0.01633477
  0.01678113 0.01578602 0.00456256 0.01444446 0.03781749 0.00600472
  0.00189015 0.00551588 0.00430037 0.00469717 0.00161942 0.00565766
  0.00526811 0.00201363 0.01081755 0.00133089 0.01849074 0.00247883
  0.00235612 0.00698906 0.00515167 0.7754795 ]
 [0.03007012 0.03225327 0.03733046 0.03260218 0.02496067 0.03048284
  0.02287731 0.01901065 0.04969597 0.05711075 0.02759367 0.02996245
  0.01544351 0.02270249 0.02899083 0.02479626 0.01228485 0.04348366
  0.09457065 0.01153575 0.04552736 0.00649154 0.02981865 0.01040036
  0.07548505 0.19823612 0.03257708 0.03360938]
 [0.00951252 0.01304991 0.05350978 0.1249356

In [19]:
from sklearn.metrics import f1_score

thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]

for t in thresholds:
    preds = (probs > t).astype(int)

    micro = f1_score(labels, preds, average="micro", zero_division=0)
    macro = f1_score(labels, preds, average="macro", zero_division=0)
    tail = tail_recall(labels, preds, tail_labels)

    print(f"Threshold: {t}")
    print(f"  Micro-F1: {micro:.4f}")
    print(f"  Macro-F1: {macro:.4f}")
    print(f"  Tail Recall: {tail:.4f}")
    print()

Threshold: 0.05
  Micro-F1: 0.3341
  Macro-F1: 0.2454
  Tail Recall: 0.2873

Threshold: 0.1
  Micro-F1: 0.4985
  Macro-F1: 0.3436
  Tail Recall: 0.1839

Threshold: 0.15
  Micro-F1: 0.5277
  Macro-F1: 0.3142
  Tail Recall: 0.1214

Threshold: 0.2
  Micro-F1: 0.5275
  Macro-F1: 0.2878
  Tail Recall: 0.0769

Threshold: 0.25
  Micro-F1: 0.5145
  Macro-F1: 0.2586
  Tail Recall: 0.0385

Threshold: 0.3
  Micro-F1: 0.4920
  Macro-F1: 0.2161
  Tail Recall: 0.0000



In [20]:
fine_thresholds = [0.05, 0.06, 0.07, 0.08, 0.09, 0.10]

print("=== Fine-grained threshold sweep ===\n")

for t in fine_thresholds:
    preds = (probs > t).astype(int)

    micro = f1_score(labels, preds, average="micro", zero_division=0)
    macro = f1_score(labels, preds, average="macro", zero_division=0)
    tail = tail_recall(labels, preds, tail_labels)

    print(f"Threshold: {t:.2f}")
    print(f"  Micro-F1: {micro:.4f}")
    print(f"  Macro-F1: {macro:.4f}")
    print(f"  Tail Recall: {tail:.4f}")
    print()

=== Fine-grained threshold sweep ===

Threshold: 0.05
  Micro-F1: 0.3341
  Macro-F1: 0.2454
  Tail Recall: 0.2873

Threshold: 0.06
  Micro-F1: 0.3791
  Macro-F1: 0.2787
  Tail Recall: 0.2716

Threshold: 0.07
  Micro-F1: 0.4099
  Macro-F1: 0.2881
  Tail Recall: 0.2151

Threshold: 0.08
  Micro-F1: 0.4469
  Macro-F1: 0.3086
  Tail Recall: 0.1839

Threshold: 0.09
  Micro-F1: 0.4778
  Macro-F1: 0.3294
  Tail Recall: 0.1839

Threshold: 0.10
  Micro-F1: 0.4985
  Macro-F1: 0.3436
  Tail Recall: 0.1839



In [21]:
# Define per-class thresholds

default_threshold = 0.1
tail_threshold = 0.05

threshold_vector = np.full(num_labels, default_threshold)
for i in tail_labels:
    threshold_vector[i] = tail_threshold

print("Threshold vector:")
print(threshold_vector)

Threshold vector:
[0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.05 0.1  0.1  0.1  0.05 0.1
 0.05 0.1  0.05 0.1  0.1  0.05 0.1  0.05 0.1  0.05 0.05 0.1  0.1  0.1 ]


In [22]:
# Apply per-class thresholds

preds = (probs > threshold_vector).astype(int)

micro = f1_score(labels, preds, average="micro", zero_division=0)
macro = f1_score(labels, preds, average="macro", zero_division=0)
tail = tail_recall(labels, preds, tail_labels)

print("=== Per-class threshold results ===")
print(f"Micro-F1: {micro:.4f}")
print(f"Macro-F1: {macro:.4f}")
print(f"Tail Recall: {tail:.4f}")

=== Per-class threshold results ===
Micro-F1: 0.4822
Macro-F1: 0.3254
Tail Recall: 0.2873


In [23]:
# Tail label support in the selected train/validation subsets

tail_label_names = {
    16: "grief",
    21: "pride",
    23: "relief",
    19: "nervousness",
    12: "embarrassment",
    24: "remorse",
    14: "fear",
    8: "desire",
}

print("Train subset tail-label support:")
for idx in tail_labels:
    count = sum(int(train_dataset[i]["labels"][idx].item()) for i in range(len(train_dataset)))
    print(f"{idx:2d} ({tail_label_names[idx]}): {count}")

print("\nValidation subset tail-label support:")
for idx in tail_labels:
    count = sum(int(val_dataset[i]["labels"][idx].item()) for i in range(len(val_dataset)))
    print(f"{idx:2d} ({tail_label_names[idx]}): {count}")

Train subset tail-label support:
16 (grief): 12
21 (pride): 9
23 (relief): 18
19 (nervousness): 23
12 (embarrassment): 28
24 (remorse): 69
14 (fear): 67
 8 (desire): 78

Validation subset tail-label support:
16 (grief): 1
21 (pride): 4
23 (relief): 1
19 (nervousness): 2
12 (embarrassment): 5
24 (remorse): 13
14 (fear): 8
 8 (desire): 8


In [24]:
# Find train examples that contain at least one tail label

tail_train_indices = []

for i in range(len(train_dataset)):
    label_vector = train_dataset[i]["labels"].numpy()
    if any(label_vector[idx] == 1 for idx in tail_labels):
        tail_train_indices.append(i)

print("Number of tail-containing train examples:", len(tail_train_indices))

Number of tail-containing train examples: 295


In [25]:
from datasets import concatenate_datasets

tail_subset = train_dataset.select(tail_train_indices)

oversampled_train_dataset = concatenate_datasets([train_dataset, tail_subset])

print("Original train size:", len(train_dataset))
print("Tail subset size:", len(tail_subset))
print("Oversampled train size:", len(oversampled_train_dataset))

Original train size: 5000
Tail subset size: 295
Oversampled train size: 5295


In [26]:
model_os = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

training_args_os = TrainingArguments(
    output_dir="./results_oversampled",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_steps=50,
    report_to="none",
)

trainer_os = Trainer(
    model=model_os,
    args=training_args_os,
    train_dataset=oversampled_train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=collator,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [27]:
trainer_os.train()
trainer_os.evaluate()

Step,Training Loss
50,0.369226
100,0.167876
150,0.162810
200,0.160135
250,0.153310
300,0.152716
350,0.144987
400,0.136564
450,0.126556
500,0.127087


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.10371033847332001,
 'eval_micro_f1': 0.44,
 'eval_macro_f1': 0.21767037485960808,
 'eval_tail_recall': 0.16466346153846154,
 'eval_runtime': 1.82,
 'eval_samples_per_second': 274.718,
 'eval_steps_per_second': 17.582,
 'epoch': 3.0}

In [28]:
pred_output_os = trainer_os.predict(val_dataset)

logits_os = pred_output_os.predictions
labels_os = pred_output_os.label_ids

probs_os = 1 / (1 + np.exp(-logits_os))

print("max prob:", probs_os.max())
print("mean prob:", probs_os.mean())

max prob: 0.9287127
mean prob: 0.04417675


In [29]:
thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]

for t in thresholds:
    preds_os = (probs_os > t).astype(int)

    micro = f1_score(labels_os, preds_os, average="micro", zero_division=0)
    macro = f1_score(labels_os, preds_os, average="macro", zero_division=0)
    tail = tail_recall(labels_os, preds_os, tail_labels)

    print(f"Threshold: {t}")
    print(f"  Micro-F1: {micro:.4f}")
    print(f"  Macro-F1: {macro:.4f}")
    print(f"  Tail Recall: {tail:.4f}")
    print()

Threshold: 0.05
  Micro-F1: 0.3625
  Macro-F1: 0.2935
  Tail Recall: 0.4938

Threshold: 0.1
  Micro-F1: 0.4968
  Macro-F1: 0.3525
  Tail Recall: 0.2969

Threshold: 0.15
  Micro-F1: 0.5375
  Macro-F1: 0.3632
  Tail Recall: 0.2812

Threshold: 0.2
  Micro-F1: 0.5497
  Macro-F1: 0.3612
  Tail Recall: 0.2464

Threshold: 0.25
  Micro-F1: 0.5573
  Macro-F1: 0.3555
  Tail Recall: 0.2464

Threshold: 0.3
  Micro-F1: 0.5470
  Macro-F1: 0.3346
  Tail Recall: 0.2464



In [30]:
fine_thresholds = [0.05, 0.06, 0.07, 0.08, 0.09, 0.10]

print("=== Fine-grained threshold sweep ===\n")

for t in fine_thresholds:
    preds = (probs > t).astype(int)

    micro = f1_score(labels, preds, average="micro", zero_division=0)
    macro = f1_score(labels, preds, average="macro", zero_division=0)
    tail = tail_recall(labels, preds, tail_labels)

    print(f"Threshold: {t:.2f}")
    print(f"  Micro-F1: {micro:.4f}")
    print(f"  Macro-F1: {macro:.4f}")
    print(f"  Tail Recall: {tail:.4f}")
    print()

=== Fine-grained threshold sweep ===

Threshold: 0.05
  Micro-F1: 0.3341
  Macro-F1: 0.2454
  Tail Recall: 0.2873

Threshold: 0.06
  Micro-F1: 0.3791
  Macro-F1: 0.2787
  Tail Recall: 0.2716

Threshold: 0.07
  Micro-F1: 0.4099
  Macro-F1: 0.2881
  Tail Recall: 0.2151

Threshold: 0.08
  Micro-F1: 0.4469
  Macro-F1: 0.3086
  Tail Recall: 0.1839

Threshold: 0.09
  Micro-F1: 0.4778
  Macro-F1: 0.3294
  Tail Recall: 0.1839

Threshold: 0.10
  Micro-F1: 0.4985
  Macro-F1: 0.3436
  Tail Recall: 0.1839



In [31]:
from sklearn.metrics import precision_score

def tail_precision(labels, preds, tail_labels):
    precisions = []
    for i in tail_labels:
        true = labels[:, i]
        pred = preds[:, i]

        tp = ((true == 1) & (pred == 1)).sum()
        fp = ((true == 0) & (pred == 1)).sum()

        if tp + fp > 0:
            precisions.append(tp / (tp + fp))

    return np.mean(precisions) if precisions else 0

In [32]:
t = 0.05
preds = (probs > t).astype(int)

tail_p = tail_precision(labels, preds, tail_labels)
tail_r = tail_recall(labels, preds, tail_labels)

print("Tail Precision:", tail_p)
print("Tail Recall:", tail_r)

Tail Precision: 0.2144308943089431
Tail Recall: 0.2872596153846154


In [33]:
from sklearn.metrics import f1_score

def tail_f1(labels, preds, tail_labels):
    f1s = []
    for i in tail_labels:
        f1 = f1_score(labels[:, i], preds[:, i], zero_division=0)
        f1s.append(f1)
    return np.mean(f1s)

print("Tail F1:", tail_f1(labels, preds, tail_labels))

Tail F1: 0.1227891156462585
